<a href="https://colab.research.google.com/github/chintu101/Cascading-Classifier-Anomaly-Detection/blob/main/Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mounting Drive

In [1]:
import numpy as np
import pandas as pd
import joblib
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/Minor_Project_Data/'
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


Loading the Models

In [7]:
model_rf      = joblib.load(SAVE_DIR + 'model_rf.pkl')
threshold_rf  = joblib.load(SAVE_DIR + 'threshold_rf.pkl')

model_xgb     = joblib.load(SAVE_DIR + 'model_xgb.pkl')
preprocessor  = joblib.load(SAVE_DIR + 'preprocessor_xgb.pkl')
threshold_xgb = joblib.load(SAVE_DIR + 'threshold_xgb.pkl')

print(f"RF  threshold loaded  : {threshold_rf}")
print(f"XGB threshold loaded  : {threshold_xgb:.4f}")

# ── Override if threshold is unreasonably high ────────────────────────────────
# Anything above 0.97 means the model will almost never flag fraud at Tier 2
# which defeats the purpose of escalating there in the first place.
# 0.50 is the standard decision boundary: above = fraud, below = legitimate.
if threshold_xgb > 0.97:
    print(f"WARNING: threshold_xgb={threshold_xgb:.4f} is too high — overriding to 0.50")
    threshold_xgb = 0.50

print(f"XGB threshold in use  : {threshold_xgb:.4f}")

RF  threshold loaded  : 0.971
XGB threshold loaded  : 0.9949
XGB threshold in use  : 0.5000


Heuristic Scorer

In [3]:
class HeuristicScorer:
    """
    Deterministic rule-based fraud suspicion scorer.
    Returns a score between 0.0 and 1.0.
    Not a probability — an ordinal suspicion index.

    Rules and weights (calibrated from EDA):
      0.55 — account fully drained after transaction
      0.20 — amount > 80% of origin balance
      0.10 — amount 50-80% of origin balance
      0.15 — destination balance inconsistency
      0.10 — transaction type is CASH_OUT or TRANSFER
      0.10 — zero-balance origin account sending > 500,000
    """

    FRAUD_TYPES           = {"CASH_OUT", "TRANSFER"}
    LARGE_AMOUNT          = 500_000

    def score(self, txn: dict) -> float:
        s = 0.0

        # Rule 1: complete account drain
        if txn["oldbalanceOrg"] > 0 and txn["newbalanceOrig"] == 0:
            s += 0.55

        # Rule 2: fraud-bearing transaction type
        if txn["type"] in self.FRAUD_TYPES:
            s += 0.10

        # Rule 3: fraction of origin balance transferred
        if txn["oldbalanceOrg"] > 0:
            ratio = txn["amount"] / (txn["oldbalanceOrg"] + 1e-9)
            if ratio > 0.80:
                s += 0.20
            elif ratio > 0.50:
                s += 0.10

        # Rule 4: destination balance inconsistency
        expected = txn["amount"]
        actual   = txn["newbalanceDest"] - txn["oldbalanceDest"]
        if abs(actual - expected) > (0.5 * txn["amount"]):
            s += 0.15

        # Rule 5: mule account pattern
        if txn["oldbalanceOrg"] == 0 and txn["amount"] > self.LARGE_AMOUNT:
            s += 0.10

        return min(s, 1.0)


heuristic = HeuristicScorer()
print("HeuristicScorer ready.")

HeuristicScorer ready.


Cascade Pipeline

In [4]:
# Thresholds for the gate zones
GREEN_THRESHOLD      = 0.15   # RF prob below this → lean legitimate
RED_THRESHOLD        = 0.80   # RF prob above this → lean fraud
DISAGREEMENT_EPSILON = 0.25   # max gap before signals are considered disagreeing
FRAUD_FREE_TYPES     = {"PAYMENT", "DEBIT", "CASH_IN"}


def run_pipeline(txn: dict) -> dict:
    """
    Run a single transaction through the full cascade pipeline.

    Parameters
    ----------
    txn : dict with keys:
        type, amount, oldbalanceOrg, newbalanceOrig,
        oldbalanceDest, newbalanceDest

    Returns
    -------
    dict with decision, tier used, probabilities, and reasoning.
    """

    # ── Tier 0: type bypass ───────────────────────────────────────────────────
    if txn["type"] in FRAUD_FREE_TYPES:
        return {
            "decision":    "LEGITIMATE",
            "tier":        0,
            "rf_prob":     0.0,
            "h_score":     0.0,
            "xgb_prob":    None,
            "reason":      f"Type '{txn['type']}' has zero historical fraud — immediate bypass.",
        }

    # ── Tier 1: RF probability ────────────────────────────────────────────────
    row        = pd.DataFrame([txn])
    rf_prob    = float(model_rf.predict_proba(row)[0, 1])

    # ── Heuristic score ───────────────────────────────────────────────────────
    h_score    = heuristic.score(txn)

    # ── Signal agreement check ────────────────────────────────────────────────
    signal_gap    = abs(rf_prob - h_score)
    signals_agree = signal_gap <= DISAGREEMENT_EPSILON

    # ── Green zone: both signals confidently say legitimate ───────────────────
    if rf_prob < GREEN_THRESHOLD and h_score < 0.20 and signals_agree:
        return {
            "decision":  "LEGITIMATE",
            "tier":      1,
            "rf_prob":   rf_prob,
            "h_score":   h_score,
            "xgb_prob":  None,
            "reason":    f"GREEN — RF={rf_prob:.3f}, H={h_score:.3f}, gap={signal_gap:.3f}. Both signals low.",
        }

    # ── Red zone: both signals confidently say fraud ──────────────────────────
    if rf_prob > RED_THRESHOLD and h_score > 0.50 and signals_agree:
        return {
            "decision":  "FRAUD",
            "tier":      1,
            "rf_prob":   rf_prob,
            "h_score":   h_score,
            "xgb_prob":  None,
            "reason":    f"RED — RF={rf_prob:.3f}, H={h_score:.3f}, gap={signal_gap:.3f}. Both signals high.",
        }

    # ── Amber zone: disagreement or uncertainty → escalate to Tier 2 ─────────
    X_prep   = preprocessor.transform(row)
    xgb_prob = float(model_xgb.predict_proba(X_prep)[0, 1])
    decision = "FRAUD" if xgb_prob >= threshold_xgb else "LEGITIMATE"

    reason = (
        f"AMBER — RF={rf_prob:.3f}, H={h_score:.3f}, gap={signal_gap:.3f} "
        f"({'disagree' if not signals_agree else 'uncertainty band'}) "
        f"→ XGB={xgb_prob:.3f} → {decision}"
    )

    return {
        "decision":  decision,
        "tier":      2,
        "rf_prob":   rf_prob,
        "h_score":   h_score,
        "xgb_prob":  xgb_prob,
        "reason":    reason,
    }


print("Pipeline function ready.")

Pipeline function ready.


Output Formatting

In [5]:
def print_result(txn: dict, result: dict):
    tier_labels = {0: "Tier 0 (type bypass)",
                   1: "Tier 1 (RF + Heuristic)",
                   2: "Tier 2 (XGBoost)"}

    verdict_color = "\033[91m" if result["decision"] == "FRAUD" else "\033[92m"
    reset         = "\033[0m"

    print("\n" + "="*55)
    print(f"  Transaction Summary")
    print("="*55)
    print(f"  Type            : {txn['type']}")
    print(f"  Amount          : {txn['amount']:,.2f}")
    print(f"  Origin balance  : {txn['oldbalanceOrg']:,.2f} → {txn['newbalanceOrig']:,.2f}")
    print(f"  Dest balance    : {txn['oldbalanceDest']:,.2f} → {txn['newbalanceDest']:,.2f}")
    print("-"*55)
    print(f"  RF probability  : {result['rf_prob']:.4f}")
    print(f"  Heuristic score : {result['h_score']:.4f}")
    if result["xgb_prob"] is not None:
        print(f"  XGB probability : {result['xgb_prob']:.4f}")
    print(f"  Decided by      : {tier_labels[result['tier']]}")
    print(f"  Routing reason  : {result['reason']}")
    print("-"*55)
    print(f"  DECISION : {verdict_color}{result['decision']}{reset}")
    print("="*55)


print("Output formatter ready.")

Output formatter ready.


Single Input Testing

In [14]:
# ── Edit these values to test different transactions ──────────────────────────
transaction = {
    "type":           "CASH_OUT",   # CASH_OUT, TRANSFER, PAYMENT, CASH_IN, DEBIT
    "amount":           20128.0,
    "oldbalanceOrg":    20128.0,
    "newbalanceOrig":   0.0,
    "oldbalanceDest":   0.0,
    "newbalanceDest":   0.0,
}

result = run_pipeline(transaction)
print_result(transaction, result)


  Transaction Summary
  Type            : CASH_OUT
  Amount          : 20,128.00
  Origin balance  : 20,128.00 → 0.00
  Dest balance    : 0.00 → 0.00
-------------------------------------------------------
  RF probability  : 0.7168
  Heuristic score : 1.0000
  XGB probability : 0.9259
  Decided by      : Tier 2 (XGBoost)
  Routing reason  : AMBER — RF=0.717, H=1.000, gap=0.283 (disagree) → XGB=0.926 → FRAUD
-------------------------------------------------------
  DECISION : FRAUD


In [10]:
transactions = [
    # Textbook fraud — full drain, CASH_OUT
    {"type": "CASH_OUT", "amount": 200_000, "oldbalanceOrg": 200_000,
     "newbalanceOrig": 0, "oldbalanceDest": 50_000, "newbalanceDest": 50_000},

    # Clear legitimate — small amount, balance intact
    {"type": "CASH_OUT", "amount": 1_000, "oldbalanceOrg": 50_000,
     "newbalanceOrig": 49_000, "oldbalanceDest": 10_000, "newbalanceDest": 11_000},

    # Type bypass — PAYMENT never has fraud
    {"type": "PAYMENT", "amount": 500, "oldbalanceOrg": 5_000,
     "newbalanceOrig": 4_500, "oldbalanceDest": 0, "newbalanceDest": 0},

    # Ambiguous — partial drain, signals may disagree
    {"type": "TRANSFER", "amount": 95_000, "oldbalanceOrg": 200_000,
     "newbalanceOrig": 105_000, "oldbalanceDest": 30_000, "newbalanceDest": 125_000},
]

for i, txn in enumerate(transactions, 1):
    print(f"\nTransaction {i} of {len(transactions)}")
    result = run_pipeline(txn)
    print_result(txn, result)


Transaction 1 of 4

  Transaction Summary
  Type            : CASH_OUT
  Amount          : 200,000.00
  Origin balance  : 200,000.00 → 0.00
  Dest balance    : 50,000.00 → 50,000.00
-------------------------------------------------------
  RF probability  : 0.6490
  Heuristic score : 1.0000
  XGB probability : 0.9712
  Decided by      : Tier 2 (XGBoost)
  Routing reason  : AMBER — RF=0.649, H=1.000, gap=0.351 (disagree) → XGB=0.971 → FRAUD
-------------------------------------------------------
  DECISION : FRAUD

Transaction 2 of 4

  Transaction Summary
  Type            : CASH_OUT
  Amount          : 1,000.00
  Origin balance  : 50,000.00 → 49,000.00
  Dest balance    : 10,000.00 → 11,000.00
-------------------------------------------------------
  RF probability  : 0.0000
  Heuristic score : 0.1000
  Decided by      : Tier 1 (RF + Heuristic)
  Routing reason  : GREEN — RF=0.000, H=0.100, gap=0.100. Both signals low.
-------------------------------------------------------
  DECISIO